In [ ]:
from flask import Flask, request, Response
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt
import time
import pandas
import random

In [ ]:
app = Flask("app")

ab = [1, 1]
total = [2, 2]
cookie = "0"

@app.route("/")
def home():
    global ab, total, cookie
    if random.random() < 0.5:
        total[0] = total[0] + 1
        resp = Response('Cat! <a href="visit?from=cat">Visit</a>')
    else:
        total[1] = total[1] + 1
        resp = Response('Dog! <a href="visit?from=dog">Visit</a>')
    resp.set_cookie("visit", "0")
    return resp

@app.route("/visit")
def test():
    global ab, total, cookie
    cookie = request.cookies.get("visit", "0")
    if cookie == "0" and request.args:
        if request.args["from"] == "cat":
            ab[0] = ab[0] + 1
        else:
            ab[1] = ab[1] + 1
    df = pandas.DataFrame([[ab[0], total[0] - ab[0]], [ab[1], total[1] - ab[1]]], columns = ["click", "no click"], index = ["cat", "dog"])
    resp = Response("The p-value is " + str(fisher_exact(df).pvalue) + "<br>" + df.to_html())
    resp.set_cookie("visit", "1")
    return resp

app.run(host = "0.0.0.0", debug = False, threaded = False)